In [1]:
from __future__ import annotations

import json
import os
import time
from pathlib import Path

_TMP_D = Path(r'D:\projeto_placentas_clayton\temp_ml')
_TMP_D.mkdir(parents=True, exist_ok=True)
os.environ['TEMP'] = str(_TMP_D)
os.environ['TMP'] = str(_TMP_D)
os.environ['TMPDIR'] = str(_TMP_D)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

_ULTRA_SETTINGS = Path(os.environ.get('APPDATA', '')) / 'Ultralytics' / 'settings.json'
if _ULTRA_SETTINGS.is_file():
    _cfg = json.loads(_ULTRA_SETTINGS.read_text(encoding='utf-8'))
    _cfg['datasets_dir'] = r'D:\projeto_placentas_clayton\datasets_ultralytics'
    _cfg['weights_dir'] = r'D:\projeto_placentas_clayton\weights_ultralytics'
    _cfg['runs_dir'] = r'D:\projeto_placentas_clayton\runs_ultralytics'
    _ULTRA_SETTINGS.write_text(json.dumps(_cfg, indent=2), encoding='utf-8')

import cv2
import numpy as np
import pandas as pd
import torch
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

print('TEMP', _TMP_D)
print('torch', torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

TEMP D:\projeto_placentas_clayton\temp_ml
torch 2.5.1+cu121 NVIDIA GeForce GTX 1650 SUPER


In [2]:
def discover_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    while p != p.parent:
        if (p / '.git').exists() and (p / 'v3_capilar_yolo11s').exists():
            return p
        p = p.parent
    raise RuntimeError('repo root not found')

REPO_ROOT = discover_repo_root()

CFG = {
    'best_pt': 'v3_capilar_yolo11s/runs/capilar_yolo11s_tiled_v4/weights/best.pt',
    'fov_root': Path(r'D:/projeto_placentas_clayton/datasat_v3.1_yolo11_og_size'),
    'output_root': 'v3_capilar_yolo11s/artifacts_v4_field',
    'imgsz': 640,
    'slice_w': 1380,
    'slice_h': 1032,
    'overlap': 0.2,
    'conf': 0.33,
    'conf_extra': [],
    'iou_match': 0.5,
    'capilar_cls': 0,
    'area_factor': (50 / 72) ** 2 * (640 / 4140) ** 2,
}

best_pt = (REPO_ROOT / CFG['best_pt']).resolve()
fov_images = CFG['fov_root'] / 'valid' / 'images'
fov_labels = CFG['fov_root'] / 'valid' / 'labels'
out_root = (REPO_ROOT / CFG['output_root']).resolve()
out_bench = out_root / 'benchmarks'
out_reports = out_root / 'reports'
out_viz = out_root / 'iou_viz'
for p in (out_root, out_bench, out_reports, out_viz):
    p.mkdir(parents=True, exist_ok=True)

val_fovs = sorted(fov_images.glob('*.jpg'))
print('best.pt', best_pt, best_pt.is_file())
print('n FOV', len(val_fovs), 'ex', val_fovs[0].name if val_fovs else None)
print('out', out_root)
assert best_pt.is_file()
assert len(val_fovs) == 27

best.pt D:\projeto_placentas_clayton\dev\projeto-placentas\v3_capilar_yolo11s\runs\capilar_yolo11s_tiled_v4\weights\best.pt True
n FOV 27 ex ROSILHA-M-B_011_jpg.rf.4a966541cdff84d474cd4516dd35b742.jpg
out D:\projeto_placentas_clayton\dev\projeto-placentas\v3_capilar_yolo11s\artifacts_v4_field


In [3]:
CAP = CFG['capilar_cls']


def parse_gt_masks(label_path: Path, w: int, h: int, cls_keep: int = CAP):
    masks = []
    if not label_path.exists():
        return masks
    for line in label_path.read_text(encoding='utf-8').splitlines():
        parts = line.strip().split()
        if len(parts) < 7:
            continue
        if int(float(parts[0])) != cls_keep:
            continue
        coords = np.array([float(x.replace(',', '.')) for x in parts[1:]], dtype=np.float32).reshape(-1, 2)
        coords[:, 0] *= w
        coords[:, 1] *= h
        m = np.zeros((h, w), dtype=np.uint8)
        cv2.fillPoly(m, [coords.astype(np.int32)], 1)
        masks.append(m)
    return masks


def pred_masks_from_sahi(result, w: int, h: int, cls_keep: int = CAP):
    out = []
    for op in result.object_prediction_list:
        if int(op.category.id) != cls_keep:
            continue
        if op.mask is None:
            continue
        m = op.mask.bool_mask.astype(np.uint8)
        if m.shape[0] != h or m.shape[1] != w:
            m = cv2.resize(m, (w, h), interpolation=cv2.INTER_NEAREST)
        out.append(m)
    return out


def iou(a: np.ndarray, b: np.ndarray) -> float:
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return float(inter) / float(union) if union else 0.0


def greedy_match(pred_masks, gt_masks, thr: float):
    scale = 4
    ps = [cv2.resize(m, (m.shape[1] // scale, m.shape[0] // scale), interpolation=cv2.INTER_NEAREST) for m in pred_masks]
    gs = [cv2.resize(m, (m.shape[1] // scale, m.shape[0] // scale), interpolation=cv2.INTER_NEAREST) for m in gt_masks]
    pairs = []
    for i, pm in enumerate(ps):
        for j, gm in enumerate(gs):
            s = iou(pm, gm)
            if s >= thr:
                pairs.append((s, i, j))
    pairs.sort(reverse=True)
    used_p, used_g, matched = set(), set(), []
    for s, i, j in pairs:
        if i in used_p or j in used_g:
            continue
        used_p.add(i)
        used_g.add(j)
        matched.append((s, i, j))
    return matched


def save_fov_viz(bgr, gt, pred, outp: Path):
    h, w = bgr.shape[:2]
    overlay = bgr.copy()
    g = np.zeros((h, w), dtype=np.uint8)
    p = np.zeros((h, w), dtype=np.uint8)
    for m in gt:
        g = np.maximum(g, m)
    for m in pred:
        p = np.maximum(p, m)
    overlay[g > 0] = (0.5 * overlay[g > 0] + np.array([0, 180, 0]) * 0.5).astype(np.uint8)
    overlay[p > 0] = (0.5 * overlay[p > 0] + np.array([0, 0, 180]) * 0.5).astype(np.uint8)
    both = np.logical_and(g > 0, p > 0)
    overlay[both] = (0.4 * overlay[both] + np.array([0, 200, 200]) * 0.6).astype(np.uint8)
    vis = np.hstack([bgr, overlay])
    scale = 1280 / vis.shape[1]
    vis = cv2.resize(vis, (1280, int(vis.shape[0] * scale)), interpolation=cv2.INTER_AREA)
    cv2.imwrite(str(outp), vis)


def sahi_on_path(det_model, img_path: Path):
    return get_sliced_prediction(
        str(img_path),
        det_model,
        slice_height=CFG['slice_h'],
        slice_width=CFG['slice_w'],
        overlap_height_ratio=CFG['overlap'],
        overlap_width_ratio=CFG['overlap'],
        perform_standard_pred=False,
        postprocess_type='GREEDYNMM',
        postprocess_match_metric='IOS',
        postprocess_match_threshold=0.5,
        exclude_classes_by_id=[1],
        verbose=0,
    )


def eval_at_conf(det_model, conf: float, save_rows: bool = True):
    det_model.confidence_threshold = conf
    tp = fp = fn = 0
    ious = []
    gt_area = pred_area = 0
    per_fov = []
    for img_path in val_fovs:
        im = cv2.imread(str(img_path))
        h, w = im.shape[:2]
        gt = parse_gt_masks(fov_labels / f'{img_path.stem}.txt', w, h)
        pred = pred_masks_from_sahi(sahi_on_path(det_model, img_path), w, h)
        matched = greedy_match(pred, gt, CFG['iou_match'])
        m = len(matched)
        fpi = len(pred) - m
        fni = len(gt) - m
        tp += m
        fp += fpi
        fn += fni
        ious.extend([s for s, _, _ in matched])
        ga = int(sum(int(x.sum()) for x in gt))
        pa = int(sum(int(x.sum()) for x in pred))
        gt_area += ga
        pred_area += pa
        prec_i = m / (m + fpi) if (m + fpi) else 0.0
        rec_i = m / (m + fni) if (m + fni) else 0.0
        f1_i = (2 * prec_i * rec_i / (prec_i + rec_i)) if (prec_i + rec_i) else 0.0
        iou_i = float(np.mean([s for s, _, _ in matched])) if matched else 0.0
        per_fov.append({
            'FOV': img_path.stem,
            'GT_Count': len(gt),
            'AI_Count': len(pred),
            'Matched': m,
            'FP': fpi,
            'FN': fni,
            'precision': prec_i,
            'recall': rec_i,
            'f1': f1_i,
            'mean_iou': iou_i,
            'GT_Area_px': ga,
            'AI_Area_px': pa,
            'GT_Area_um2': ga * CFG['area_factor'],
            'AI_Area_um2': pa * CFG['area_factor'],
            'Area_Diff_pct': (100.0 * (pa - ga) / ga) if ga else 0.0,
        })
        if save_rows:
            save_fov_viz(im, gt, pred, out_viz / f'iou_viz_{img_path.stem}.png')
        print(img_path.name, 'gt', len(gt), 'pred', len(pred), 'tp', m, 'f1', round(f1_i, 3), flush=True)
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) else 0.0
    mean_iou = float(np.mean(ious)) if ious else 0.0
    area_rel_err = abs(pred_area - gt_area) / gt_area if gt_area else 0.0
    summary = {
        'conf': conf,
        'tp': tp, 'fp': fp, 'fn': fn,
        'precision': prec, 'recall': rec, 'f1': f1,
        'mean_iou': mean_iou,
        'gt_area_px': gt_area, 'pred_area_px': pred_area,
        'gt_area_um2': gt_area * CFG['area_factor'],
        'pred_area_um2': pred_area * CFG['area_factor'],
        'area_rel_error': area_rel_err,
        'score': 0.6 * f1 + 0.3 * mean_iou + 0.1 * (1.0 - area_rel_err),
        'n_fov': len(val_fovs),
    }
    if save_rows:
        pd.DataFrame(per_fov).to_csv(out_reports / 'capilar_field_totals_report.csv', index=False)
    return summary, per_fov

print('helpers ok')

helpers ok


In [4]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
det = AutoDetectionModel.from_pretrained(
    model_type='ultralytics',
    model_path=str(best_pt),
    confidence_threshold=CFG['conf'],
    device=device,
    image_size=CFG['imgsz'],
    task='segment',
)
print('sahi model', device, 'imgsz', CFG['imgsz'], 'slice', CFG['slice_w'], CFG['slice_h'], 'overlap', CFG['overlap'])

t0 = time.time()
confs = [CFG['conf']] + list(CFG['conf_extra'])
rows = []
best_summary = None
best_per_fov = None
for conf in confs:
    summary, per_fov = eval_at_conf(det, conf, save_rows=(conf == CFG['conf']))
    rows.append(summary)
    if conf == CFG['conf']:
        best_summary, best_per_fov = summary, per_fov
    print('CONF', conf, 'F1', round(summary['f1'], 4), 'IoU', round(summary['mean_iou'], 4), 'area_err', round(summary['area_rel_error'], 4), flush=True)

sweep_df = pd.DataFrame(rows).sort_values('score', ascending=False)
sweep_df.to_csv(out_bench / 'validation_conf_sweep_field.csv', index=False)
selected = {
    'model': 'yolo11s-seg',
    'run_name': 'capilar_yolo11s_tiled_v4',
    'checkpoint': str(best_pt),
    'eval': 'sahi_field',
    'slice': [CFG['slice_w'], CFG['slice_h']],
    'overlap': CFG['overlap'],
    'perform_standard_pred': False,
    **{k: best_summary[k] for k in ['conf', 'f1', 'mean_iou', 'area_rel_error', 'precision', 'recall', 'tp', 'fp', 'fn', 'n_fov']},
    'best_conf': best_summary['conf'],
    'weighted_score': best_summary['score'],
    'area_factor_um2_per_px2': CFG['area_factor'],
}
(out_bench / 'selected_confidence_field.json').write_text(json.dumps(selected, indent=2), encoding='utf-8')
print(json.dumps(selected, indent=2))
print('elapsed_s', round(time.time() - t0, 1))

d:\MINICONDA3\ENVS\PROJETO_PLACENTAS\lib\site-packages\torch\nn\modules\module.py:1326: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  return t.to(


sahi model cuda:0 imgsz 640 slice 1380 1032 overlap 0.2
ROSILHA-M-B_011_jpg.rf.4a966541cdff84d474cd4516dd35b742.jpg gt 115 pred 107 tp 87 f1 0.784
ROSILHA-M-D_006_jpg.rf.4bf38cf5314617b13c7c11445fe40b04.jpg gt 200 pred 198 tp 173 f1 0.869
ROSILHA-M-D_011_jpg.rf.3a2fb2d36de8bcc90049d66dd562a41c.jpg gt 226 pred 206 tp 195 f1 0.903
TORDILHA-B_006_jpg.rf.235faef65a5565236d3b0050d4713580.jpg gt 279 pred 241 tp 198 f1 0.762
TORDILHA-D_003_jpg.rf.e67cd4ad20c15ed213012bb43fa0dc3f.jpg gt 212 pred 279 tp 129 f1 0.525
TORDILHA-D_011_jpg.rf.67cbd65b708b681788cae0003256ca67.jpg gt 238 pred 219 tp 187 f1 0.818
TORDILHA-G_012_jpg.rf.5a71e85cb1442ba538dc7acb7149a07d.jpg gt 257 pred 232 tp 200 f1 0.818
TOSTADA-B_004_jpg.rf.25c22e0051034e43662e42875d5b085f.jpg gt 119 pred 100 tp 65 f1 0.594
TOSTADA-B_011_jpg.rf.e52fc7a5f2ca4041d6a049b2f927f63d.jpg gt 211 pred 114 tp 78 f1 0.48
TOSTADA-D_004_jpg.rf.3e98f483ac7b358f6b42724649248703.jpg gt 132 pred 93 tp 59 f1 0.524
TOSTADA-D_005_jpg.rf.9eda8c127b0d32eddbe

In [5]:
pngs = sorted(out_viz.glob('iou_viz_*.png'))
print('n viz', len(pngs), 'expected', len(val_fovs))
print(out_viz)
for p in pngs:
    print(p.name)

n viz 27 expected 27
D:\projeto_placentas_clayton\dev\projeto-placentas\v3_capilar_yolo11s\artifacts_v4_field\iou_viz
iou_viz_ROSILHA-M-B_011_jpg.rf.4a966541cdff84d474cd4516dd35b742.png
iou_viz_ROSILHA-M-D_006_jpg.rf.4bf38cf5314617b13c7c11445fe40b04.png
iou_viz_ROSILHA-M-D_011_jpg.rf.3a2fb2d36de8bcc90049d66dd562a41c.png
iou_viz_TORDILHA-B_006_jpg.rf.235faef65a5565236d3b0050d4713580.png
iou_viz_TORDILHA-D_003_jpg.rf.e67cd4ad20c15ed213012bb43fa0dc3f.png
iou_viz_TORDILHA-D_011_jpg.rf.67cbd65b708b681788cae0003256ca67.png
iou_viz_TORDILHA-G_012_jpg.rf.5a71e85cb1442ba538dc7acb7149a07d.png
iou_viz_TOSTADA-B_004_jpg.rf.25c22e0051034e43662e42875d5b085f.png
iou_viz_TOSTADA-B_011_jpg.rf.e52fc7a5f2ca4041d6a049b2f927f63d.png
iou_viz_TOSTADA-D_004_jpg.rf.3e98f483ac7b358f6b42724649248703.png
iou_viz_TOSTADA-D_005_jpg.rf.9eda8c127b0d32eddbe6f3379d28ec98.png
iou_viz_TOSTADA-G_004_jpg.rf.a8373c1bf8f08d538788901740142a7d.png
iou_viz_TOSTADA-G_011_jpg.rf.9ccdeb5753c9cd5d19a9f9b7f930b630.png
iou_viz_TP-D_0